# The head-direction system is a continuous ring attractor whose structure is internally maintained during sleep

**Dataset**: DANDI Archive dandiset
[000056](https://dandiarchive.org/dandiset/000056) (Peyrache et al. 2015,
*Nature Neuroscience*, "Internally organized mechanisms of the head direction
sense"). Extracellular recordings from the anterodorsal thalamus (ADn) /
postsubiculum of mice, with dual-LED head tracking and manually scored
wake / Non-REM / REM states.

**Question**. The head-direction (HD) system is the classic candidate for a
*continuous ring attractor*: a network whose stable states form a ring, so
that population activity is always a single "bump" whose position encodes
the animal's heading. The decisive test of the attractor hypothesis is that
the ring organization does not depend on sensory input: if the bump is
generated by internal recurrent dynamics, the pairwise correlation structure
of HD cells measured during wake must persist during sleep, and the
population activity during sleep must still form a localized bump that
drifts smoothly along the ring.

**Approach** (all analyses use Pynapple on data streamed from DANDI with
remfile; no file is downloaded in full):
1. Extract the HD angle from the two LEDs and identify HD cells from their
   wake tuning curves (mean vector length + a random-time resampling null).
2. During wake, show that the population forms one bump that tracks the true
   heading, and that heading can be Bayesian-decoded from 100 ms population
   vectors.
3. During sleep, show that the wake correlation structure is preserved
   (the main result of Peyrache et al. 2015), with a label-permutation null.
4. Decode a *virtual* heading during REM sleep from the wake tuning curves
   and show the bump is internally maintained: localized, slowly drifting,
   covering the ring, and significantly more temporally coherent than a
   circular time-shift control that preserves firing rates but destroys
   cross-cell coordination.
5. Repeat the key analyses on four more sessions (one per mouse) and pool.

A technical detail: the Bayesian decoder's Poisson normalization term
`exp(-dt * sum_i rate_i(x))` acts as a static bias toward directions where
the recorded population's total rate is low. Because preferred directions
and rates are not perfectly uniform, we flatten this "rate landscape" by
rescaling each cell's tuning curve (non-negative least squares) before
decoding. The same flattened curves are used for wake and sleep decoding.

In [ ]:
import numpy as np
import h5py
import remfile
import xarray as xr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pynapple as nap
from pynwb import NWBHDF5IO
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import nnls
from tqdm import tqdm

rng = np.random.default_rng(0)

DANDI_VERSION = "0.250624.0430"
# One session per mouse, chosen for file size and HD-cell yield.
SESSIONS = {
    "Mouse28-140310": "656704ea-a4cd-40f4-8158-a6533ebf2eee",
    "Mouse25-140123": "bdb30f7d-ba69-4d2e-8504-2efab69cd8d7",
    "Mouse17-130128": "4cc64fe0-7b1e-404c-8b86-fb5659292830",
    "Mouse20-130514": "748aa5de-c0de-4aa7-a7ef-2aad2f87a7eb",
    "Mouse24-131213": "ada02790-6eb6-48ee-902d-9ba017303586",
}
DISK_CACHE = remfile.DiskCache("/tmp/remfile_cache_hd")


def load_session(asset_id):
    """Stream an NWB session from DANDI via remfile (follows the redirect)."""
    url = f"https://api.dandiarchive.org/api/assets/{asset_id}/download/"
    rem_file = remfile.File(url, disk_cache=DISK_CACHE)
    h5py_file = h5py.File(rem_file, "r")
    io = NWBHDF5IO(file=h5py_file)
    return nap.NWBFile(io.read())


def get_hd_angle(nwb):
    """HD from the two LEDs; tracking failures are sentinel -1 -> NaN."""
    red = nwb["SubjectPosition/RedLED"]
    blue = nwb["SubjectPosition/BlueLED"]
    red_xy = np.asarray(red.values)
    blue_xy = np.asarray(blue.values)
    good = (red_xy > 0).all(axis=1) & (blue_xy > 0).all(axis=1)
    ang = np.arctan2(red_xy[:, 1] - blue_xy[:, 1],
                     red_xy[:, 0] - blue_xy[:, 0]) % (2 * np.pi)
    ang[~good] = np.nan
    return nap.Tsd(t=red.t, d=ang)


def get_states(nwb):
    states = nwb["states"]
    return {lbl: states[states["label"] == lbl] for lbl in np.unique(states["label"])}


def sorted_units(nwb):
    """Units as a TsGroup with sorted spike times (one unit in Mouse17 is unsorted)."""
    units = nwb["units"]
    return nap.TsGroup({k: nap.Ts(np.sort(units[k].t)) for k in units.keys()})


def mean_vector_length(angles):
    return np.sqrt(np.sum(np.sin(angles)) ** 2 + np.sum(np.cos(angles)) ** 2) / len(angles)


def flat_tuning(rates):
    """Scale tuning curves (NNLS, w >= 0) so the population rate landscape
    sum_i w_i * rate_i(x) is ~constant across directions."""
    tc = np.nan_to_num(rates, nan=0.0)
    w, _ = nnls(tc.T, np.full(tc.shape[1], tc.sum(axis=0).mean()))
    return tc * w[:, None], w


def time_shift_group(group, epochs, rng):
    """Circularly shift each unit's spikes by a random offset within each epoch:
    preserves firing rates and single-epoch rate structure, destroys cross-cell
    coordination."""
    data = {}
    for k in group.keys():
        pieces = []
        for st, en in zip(epochs.start, epochs.end):
            tt = group[k].t[(group[k].t >= st) & (group[k].t < en)]
            L = en - st
            if len(tt) and L > 0:
                tt = st + (tt - st + rng.uniform(0, L)) % L
            pieces.append(tt)
        data[k] = nap.Ts(np.sort(np.concatenate(pieces)))
    return nap.TsGroup(data)

## 1. Load the main session and validate the raw data

Mouse28-140310 (47 units, ~3 h wake, ~2.7 h Non-REM, ~16 min REM).
The HD angle covers the full circle; spike rasters show heterogeneous,
stable firing.

In [ ]:
name = "Mouse28-140310"
nwb = load_session(SESSIONS[name])
print(nwb)
units = sorted_units(nwb)
hd = get_hd_angle(nwb)
states = get_states(nwb)
wake = states["Awake"]
print(f"{len(units)} units; wake {wake.tot_length('s'):.0f} s, "
      f"NREM {states['Non-REM'].tot_length('s'):.0f} s, "
      f"REM {states['REM'].tot_length('s'):.0f} s")
print(f"HD: {len(hd)} samples at ~{1/np.median(np.diff(hd.t)):.1f} Hz, "
      f"{np.isnan(hd.values).mean()*100:.1f}% tracking failures")

fig, axes = plt.subplots(3, 1, figsize=(12, 8), gridspec_kw=dict(hspace=0.55))
ep = nap.IntervalSet(start=wake.start[0], end=wake.start[0] + 60)
hd_ep = hd.restrict(ep)
axes[0].plot(hd_ep.t, hd_ep.values, ".", ms=1, color="k")
axes[0].set_ylabel("HD (rad)")
axes[0].set_ylim(-0.2, 2 * np.pi + 0.2)
axes[0].set_title("Head-direction angle from LED difference (first 60 s of wake)")

hd_wake = hd.restrict(wake)
occ, edges = np.histogram(hd_wake.values[~np.isnan(hd_wake.values)], bins=60,
                          range=(0, 2 * np.pi))
axes[1].plot(np.degrees(0.5 * (edges[:-1] + edges[1:])), occ / occ.sum(), color="k")
axes[1].set_ylabel("occupancy")
axes[1].set_xlabel("HD (deg)")
axes[1].set_title("Wake HD occupancy covers the whole circle")

ep2 = nap.IntervalSet(start=wake.start[0], end=wake.start[0] + 30)
spk = units.restrict(ep2)
for i, k in enumerate(units.keys()):
    axes[2].plot(spk[k].t, np.full(len(spk[k].t), i), "|", ms=2, color="k")
axes[2].set_ylabel("unit #")
axes[2].set_xlabel("time (s)")
axes[2].set_title("Spike raster (30 s of wake)")
fig.suptitle(f"{name}: raw data validation", fontsize=13)
fig.savefig("figures/01_raw_data_validation.png", dpi=150, bbox_inches="tight")

## 2. HD tuning curves and HD cell identification

Tuning curves are spike counts per 6-degree bin divided by occupancy. A unit
is an HD cell if its spike-angle mean vector length (MVL) exceeds 0.3 and is
significant against a random-time null (spike counts redrawn from the wake
HD occupancy; 1000 shuffles), which correctly accounts for non-uniform
heading occupancy.

In [ ]:
counts_da = nap.compute_tuning_curves(units, hd, bins=60, range=[(0, 2 * np.pi)],
                                      epochs=wake, return_counts=True,
                                      feature_names=["hd"])
counts = counts_da.values
occupancy = counts_da.attrs["occupancy"] / counts_da.attrs["fs"]
rates = np.where(occupancy[None, :] > 0, counts / occupancy[None, :], np.nan)
bin_centers = counts_da.coords["hd"].values

hd_valid_v = hd_wake.values[~np.isnan(hd_wake.values)]
N_SHUFFLE_HD, MVL_FLOOR, SPIKE_CAP = 1000, 0.3, 100_000
n_units = len(units)
mvl_obs = np.zeros(n_units)
p_val = np.ones(n_units)
pref = np.zeros(n_units)
for i, k in enumerate(tqdm(units.keys(), desc="HD stats")):
    spk = units[k].restrict(wake)
    if len(spk) == 0:
        continue
    ang = spk.value_from(hd)
    ang = ang[~np.isnan(ang)]
    if len(ang) > SPIKE_CAP:
        ang = ang[rng.choice(len(ang), SPIKE_CAP, replace=False)]
    if len(ang) < 50:
        continue
    mvl_obs[i] = mean_vector_length(ang)
    pref[i] = np.arctan2(np.mean(np.sin(ang)), np.mean(np.cos(ang))) % (2 * np.pi)
    n = len(ang)
    null = np.zeros(N_SHUFFLE_HD)
    for cs in range(0, N_SHUFFLE_HD, 100):
        m = min(100, N_SHUFFLE_HD - cs)
        samp = hd_valid_v[rng.integers(0, len(hd_valid_v), size=(m, n))]
        null[cs:cs + m] = np.sqrt(np.sin(samp).mean(1) ** 2 + np.cos(samp).mean(1) ** 2)
    p_val[i] = (np.sum(null >= mvl_obs[i]) + 1) / (N_SHUFFLE_HD + 1)

is_hd = (p_val < 0.05) & (mvl_obs > MVL_FLOOR)
n_hd = int(is_hd.sum())
print(f"HD cells: {n_hd}/{n_units} (MVL>{MVL_FLOOR}, random-time null p<0.05)")

hd_idx = np.where(is_hd)[0]
order = np.argsort(pref[hd_idx])
hd_sorted = hd_idx[order]
pref_sorted = pref[hd_sorted]
hd_units = nap.TsGroup({i: units[int(u)] for i, u in enumerate(hd_sorted)})
iu = np.triu_indices(n_hd, 1)

fig = plt.figure(figsize=(13, 8.5))
gs = fig.add_gridspec(2, 3, hspace=0.5, wspace=0.35)
ax = fig.add_subplot(gs[0, 0])
ax.scatter(mvl_obs[~is_hd], -np.log10(p_val[~is_hd]), s=25, c="0.6", label="non-HD")
ax.scatter(mvl_obs[is_hd], -np.log10(p_val[is_hd]), s=25, c="crimson", label="HD cell")
ax.axhline(-np.log10(0.05), ls="--", c="k", lw=0.8)
ax.axvline(MVL_FLOOR, ls="--", c="k", lw=0.8)
ax.set_xlabel("mean vector length")
ax.set_ylabel("-log10(p) vs random-time null")
ax.legend(fontsize=8)
ax.set_title("HD cell selection")

ax = fig.add_subplot(gs[0, 1:])
tc_norm = rates[hd_sorted] / np.nanmax(rates[hd_sorted], axis=1, keepdims=True)
im = ax.imshow(tc_norm, aspect="auto", cmap="viridis",
               extent=[0, 360, n_hd - 0.5, -0.5])
ax.set_xlabel("head direction (deg)")
ax.set_ylabel("HD cell (sorted by pref. dir.)")
ax.set_title("Normalized wake tuning curves tile the circle")
fig.colorbar(im, ax=ax, label="norm. rate", shrink=0.8)

ax = fig.add_subplot(gs[1, :])
ax.hist(np.degrees(pref_sorted), bins=18, range=(0, 360), color="crimson", alpha=0.7)
ax.set_xlabel("preferred direction (deg)")
ax.set_ylabel("count")
ax.set_title("Preferred directions of HD cells")
fig.suptitle(f"{name}: HD cell identification", fontsize=13)
fig.savefig("figures/02a_hd_cell_selection.png", dpi=150, bbox_inches="tight")

# polar view of every HD tuning curve
n_col = 5
n_row = int(np.ceil(n_hd / n_col))
fig, axes = plt.subplots(n_row, n_col, figsize=(13, 2.2 * n_row),
                         subplot_kw=dict(projection="polar"))
for j, ax in enumerate(axes.flat):
    if j >= n_hd:
        ax.axis("off")
        continue
    tc = rates[hd_sorted[j]]
    ax.plot(bin_centers, tc, color="crimson", lw=1.2)
    ax.fill_between(bin_centers, 0, np.nan_to_num(tc), color="crimson", alpha=0.3)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"MVL={mvl_obs[hd_sorted[j]]:.2f}", fontsize=8)
fig.suptitle(f"{name}: wake tuning curves of the {n_hd} HD cells (sorted by pref. dir.)",
             fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig("figures/02b_tuning_curves_polar.png", dpi=150, bbox_inches="tight")

## 3. Wake: one bump of activity on a ring tracks the true heading

Sorting HD cells by preferred direction reveals the ring topology directly:
at every moment the active cells form a single localized bump whose position
follows the LED-measured heading. A Bayesian decoder (Zhang et al. 1998;
`nap.decode_bayes`, 100 ms bins) reconstructs the heading from the
population with a median circular error of ~20 degrees.

In [ ]:
tc_rates, scales = flat_tuning(rates[hd_sorted])
print(f"landscape flattening: {(scales > 0.01).sum()}/{n_hd} cells used, "
      f"landscape CV {(tc_rates.sum(0).std()/tc_rates.sum(0).mean()):.3f}")
tuning_da = xr.DataArray(tc_rates, dims=("unit", "hd"),
                         coords={"unit": np.arange(n_hd), "hd": bin_centers})

decoded, posterior = nap.decode_bayes(tuning_da, hd_units, epochs=wake, bin_size=0.1)
actual = decoded.value_from(hd)
valid = ~np.isnan(actual.values) & ~np.isnan(decoded.values)
err = np.angle(np.exp(1j * (decoded.values[valid] - actual.values[valid])))
med_err = np.degrees(np.median(np.abs(err)))
print(f"median |circular error| on wake: {med_err:.1f} deg ({valid.sum()} bins)")

fig = plt.figure(figsize=(13, 10))
gs = fig.add_gridspec(3, 2, hspace=0.5, wspace=0.3, height_ratios=[1.2, 1.2, 1])
dur = wake.end - wake.start
i_long = int(np.argmax(dur))
t0 = wake.start[i_long] + (dur[i_long] - 60) / 2
ep = nap.IntervalSet(start=t0, end=t0 + 60)

counts_w = hd_units.count(0.05, ep)
sm = gaussian_filter1d(counts_w.values.astype(float), sigma=2, axis=0)
norm = sm / (sm.max(axis=1, keepdims=True) + 1e-9)
ax = fig.add_subplot(gs[0, :])
ax.imshow(norm.T, aspect="auto", cmap="viridis", origin="lower",
          extent=[counts_w.t[0], counts_w.t[-1], -0.5, n_hd - 0.5])
hd_ep = hd.restrict(ep)
ax.plot(hd_ep.t, np.interp(hd_ep.values % (2 * np.pi), pref_sorted % (2 * np.pi),
                           np.arange(n_hd), period=n_hd),
        color="red", lw=1.2, label="true HD")
ax.set_ylabel("HD cell (sorted by pref. dir.)")
ax.set_xlabel("time (s)")
ax.set_title("Population activity: a single bump moves along the ring")
ax.legend(loc="upper right", fontsize=8)

dec_ep = decoded.restrict(ep)
post_ep = posterior.restrict(ep)
ax = fig.add_subplot(gs[1, :])
P = post_ep.values.T
ax.imshow(P / (P.sum(axis=0, keepdims=True) + 1e-12), aspect="auto", cmap="magma",
          origin="lower", extent=[dec_ep.t[0], dec_ep.t[-1], 0, 360])
ax.plot(hd_ep.t, np.degrees(hd_ep.values), color="cyan", lw=1.0, label="true HD")
ax.plot(dec_ep.t, np.degrees(dec_ep.values), ".", color="white", ms=1.5,
        label="decoded HD")
ax.set_ylabel("head direction (deg)")
ax.set_xlabel("time (s)")
ax.set_title("Bayesian posterior over HD (100 ms bins, wake tuning curves)")
ax.legend(loc="upper right", fontsize=8)

ax = fig.add_subplot(gs[2, 0])
ax.plot(np.degrees(actual.values[valid]), np.degrees(decoded.values[valid]), ".",
        ms=1, alpha=0.3, color="k")
ax.plot([0, 360], [0, 360], "r--", lw=1)
ax.set_xlabel("true HD (deg)")
ax.set_ylabel("decoded HD (deg)")
ax.set_title(f"Wake decoding (median err {med_err:.0f} deg)")

ax = fig.add_subplot(gs[2, 1])
ax.hist(np.degrees(err), bins=72, range=(-180, 180), color="k")
ax.axvline(0, color="r", ls="--", lw=1)
ax.set_xlabel("circular error (deg)")
ax.set_ylabel("count")
ax.set_title("Decoding error distribution")
fig.suptitle(f"{name}: ring-attractor readout during wake", fontsize=13)
fig.savefig("figures/03_wake_ring_decoding.png", dpi=150, bbox_inches="tight")

## 4. Sleep: the pairwise correlation structure of the ring is preserved

The decisive test (Peyrache et al. 2015): if the ring is an *internal*
attractor, the pattern of pairwise correlations measured during wake must
persist in sleep, when there is no heading signal. We bin HD-cell spikes in
0.5 s bins, compute pairwise Pearson correlations per epoch, and average
across epochs with a Fisher-z weighting. Wake correlations are computed
during active head-movement periods (top-quartile angular speed), where the
bump rotates and rates co-modulate. Significance comes from a
label-permutation null (1000 shuffles of the sleep matrix rows/columns),
which tests that the *specific* pairing, i.e. the ring organization, is
preserved.

In [ ]:
CORR_BIN = 0.5
hd_w = hd.restrict(wake)
v = np.abs(np.angle(np.exp(1j * np.diff(hd_w.values)))) / np.diff(hd_w.t)
v = np.concatenate([[0], v])
v[np.isnan(v)] = 0
v_sm = gaussian_filter1d(v, sigma=39)
active_ep = nap.Tsd(t=hd_w.t, d=v_sm).threshold(
    np.percentile(v_sm, 75), "above").time_support.intersect(wake)
print(f"active wake: {active_ep.tot_length('s'):.0f} s of {wake.tot_length('s'):.0f} s")


def state_corr(epochs, label):
    zs, ws = [], []
    for st, en in zip(epochs.start, epochs.end):
        if en - st < 5.0:
            continue
        c = hd_units.count(CORR_BIN, nap.IntervalSet(st, en)).values.astype(float)
        if c.shape[0] < 20:
            continue
        with np.errstate(invalid="ignore"):
            C = np.corrcoef(c.T)
        zs.append(np.arctanh(np.clip(C[iu], -0.999, 0.999)))
        ws.append(c.shape[0])
    Z = np.stack(zs)
    W = np.broadcast_to(np.array(ws, float)[:, None], Z.shape).copy()
    W[np.isnan(Z)] = 0.0
    Z = np.nan_to_num(Z)
    den = W.sum(0)
    z_mean = np.where(den > 0, Z.sum(0) / np.maximum(den, 1e-12), np.nan)
    Cmean = np.full((n_hd, n_hd), np.nan)
    Cmean[iu] = np.tanh(z_mean)
    Cmean[(iu[1], iu[0])] = Cmean[iu]
    np.fill_diagonal(Cmean, 1.0)
    print(f"{label}: {len(zs)} epochs")
    return Cmean


C_wake = state_corr(active_ep, "active wake")
C_rem = state_corr(states["REM"], "REM")
C_nrem = state_corr(states["Non-REM"], "NREM")

pw, pr, pn = C_wake[iu], C_rem[iu], C_nrem[iu]
finite = np.isfinite(pw) & np.isfinite(pr) & np.isfinite(pn)
pw, pr, pn = pw[finite], pr[finite], pn[finite]
r_rem = np.corrcoef(pw, pr)[0, 1]
r_nrem = np.corrcoef(pw, pn)[0, 1]
print(f"corr-of-corr wake-REM: {r_rem:.3f}, wake-NREM: {r_nrem:.3f} ({finite.sum()} pairs)")

N_PERM = 1000
null_rem, null_nrem = np.zeros(N_PERM), np.zeros(N_PERM)
for i in tqdm(range(N_PERM), desc="permutation null"):
    p = rng.permutation(n_hd)
    null_rem[i] = np.corrcoef(pw, C_rem[np.ix_(p, p)][iu][finite])[0, 1]
    null_nrem[i] = np.corrcoef(pw, C_nrem[np.ix_(p, p)][iu][finite])[0, 1]
p_rem = (np.sum(null_rem >= r_rem) + 1) / (N_PERM + 1)
p_nrem = (np.sum(null_nrem >= r_nrem) + 1) / (N_PERM + 1)
print(f"permutation p: REM {p_rem:.4f}, NREM {p_nrem:.4f}")

d_ang = np.abs(np.angle(np.exp(1j * (pref_sorted[:, None] - pref_sorted[None, :]))))[iu][finite]
bins = np.linspace(0, np.pi, 7)
bin_id = np.digitize(d_ang, bins) - 1
centers = 0.5 * (bins[:-1] + bins[1:])
bw = np.array([np.nanmean(pw[bin_id == b]) for b in range(6)])
br = np.array([np.nanmean(pr[bin_id == b]) for b in range(6)])
bn = np.array([np.nanmean(pn[bin_id == b]) for b in range(6)])

## 5. Sleep: an internally generated bump drifts along the ring during REM

There is no heading signal during sleep, but if the attractor is intact we
can still ask where the bump *would* be: decode a virtual heading from the
wake tuning curves. The decoded trajectory should be (a) localized on the
ring, (b) slowly drifting rather than jumping, and (c) visiting the whole
ring over time. The control is a circular time-shift of each unit's spikes
within each REM epoch, which preserves every unit's firing rate and slow
rate fluctuations but destroys cross-cell coordination; any smoothness that
survives the shift cannot be attributed to attractor dynamics.

In [ ]:
DEC_BIN = 0.1
rem = states["REM"]
dec_rem_all, post_rem_all = nap.decode_bayes(tuning_da, hd_units, epochs=rem,
                                             bin_size=DEC_BIN)
counts_rem = hd_units.count(DEC_BIN, rem)
enough = counts_rem.values.sum(axis=1) >= 2
print(f"REM bins with >=2 spikes: {enough.mean()*100:.0f}% ({enough.sum()} bins)")


def angular_speed(dec, keep):
    t, vv = dec.t, dec.values
    dt = np.diff(t)
    good = (dt < 0.2) & ~np.isnan(vv[:-1]) & ~np.isnan(vv[1:]) & keep[:-1] & keep[1:]
    dv = np.abs(np.angle(np.exp(1j * np.diff(vv))))
    return dv[good] / dt[good]


def posterior_autocorr(post, keep, max_lag=50):
    P = post.values
    Pn = P / (P.sum(axis=1, keepdims=True) + 1e-12)
    t = post.t
    out = np.full(max_lag + 1, np.nan)
    for k in range(1, max_lag + 1):
        dt = t[k:] - t[:-k]
        ok = (np.abs(dt - k * DEC_BIN) < 0.02) & keep[k:] & keep[:-k]
        if ok.sum() < 10:
            continue
        a, b = Pn[:-k][ok], Pn[k:][ok]
        out[k] = np.mean(np.sum(a * b, 1) /
                         (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-12))
    return out


sp_real = angular_speed(dec_rem_all, enough)
ac_real = posterior_autocorr(post_rem_all, enough)
p90_real = np.percentile(sp_real, 90)
print(f"real REM: p90 angular speed {p90_real:.0f} deg/s, "
      f"autocorr lag-1s {ac_real[10]:.3f}")

N_SHIFT = 100
shift_p90 = np.zeros(N_SHIFT)
shift_ac = np.full((N_SHIFT, 51), np.nan)
for i in tqdm(range(N_SHIFT), desc="time-shift null"):
    shifted = time_shift_group(hd_units, rem, rng)
    dec_s, post_s = nap.decode_bayes(tuning_da, shifted, epochs=rem, bin_size=DEC_BIN)
    keep_s = shifted.count(DEC_BIN, rem).values.sum(axis=1) >= 2
    shift_p90[i] = np.percentile(angular_speed(dec_s, keep_s), 90)
    shift_ac[i] = posterior_autocorr(post_s, keep_s)
p_speed = (np.sum(shift_p90 <= p90_real) + 1) / (N_SHIFT + 1)
p_ac10 = (np.sum(shift_ac[:, 10] >= ac_real[10]) + 1) / (N_SHIFT + 1)
print(f"time-shift null p90 speed {np.median(shift_p90):.0f} deg/s (real slower, p={p_speed:.4f}); "
      f"autocorr lag-1s null {np.nanmean(shift_ac[:,10]):.3f} (real higher, p={p_ac10:.4f})")


def posterior_R(post, keep):
    P = post.values[keep]
    P = P / (P.sum(axis=1, keepdims=True) + 1e-12)
    th = bin_centers[None, :]
    return np.sqrt(np.sum(P * np.sin(th), 1) ** 2 + np.sum(P * np.cos(th), 1) ** 2)


R_rem = posterior_R(post_rem_all, enough)
R_rem = R_rem[~np.isnan(R_rem)]
enough_wake = hd_units.count(DEC_BIN, wake).values.sum(axis=1) >= 2
R_wake = posterior_R(posterior, enough_wake)
R_wake = R_wake[~np.isnan(R_wake)]
print(f"posterior concentration R: wake {np.median(R_wake):.3f}, REM {np.median(R_rem):.3f}")

# example time-shifted decode + best 90 s window of the longest REM episode
shifted_ex = time_shift_group(hd_units, rem, rng)
dec_shift_ex, _ = nap.decode_bayes(tuning_da, shifted_ex, epochs=rem, bin_size=DEC_BIN)
i_rem = int(np.argmax(rem.end - rem.start))
rem_start, rem_end = rem.start[i_rem], rem.end[i_rem]
cnt_ep = counts_rem.restrict(nap.IntervalSet(rem_start, rem_end)).values.sum(axis=1)
t_ep = counts_rem.restrict(nap.IntervalSet(rem_start, rem_end)).t
win = int(90 / DEC_BIN)
w_start = t_ep[int(np.argmax(np.convolve(cnt_ep, np.ones(win), mode="valid")))] if len(cnt_ep) > win else rem_start
ep_rem = nap.IntervalSet(start=w_start, end=min(w_start + 90, rem_end))
dec_rem = dec_rem_all.restrict(ep_rem)
post_rem = post_rem_all.restrict(ep_rem)
dec_shuf_rem = dec_shift_ex.restrict(ep_rem)
enough_rem = counts_rem.restrict(ep_rem).values.sum(axis=1) >= 2

# ---- figure: correlation preservation + REM bump episode ----
fig = plt.figure(figsize=(14, 11))
gs = fig.add_gridspec(3, 3, hspace=0.55, wspace=0.4)
allc = np.concatenate([C_wake[iu], C_rem[iu], C_nrem[iu]])
vmax = np.nanpercentile(np.abs(allc[np.isfinite(allc)]), 99)
for j, (C, ttl) in enumerate([(C_wake, "wake (active)"), (C_rem, "REM"), (C_nrem, "NREM")]):
    ax = fig.add_subplot(gs[0, j])
    Cplot = C.copy()
    np.fill_diagonal(Cplot, np.nan)
    im = ax.imshow(Cplot, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(f"pairwise corr, {ttl}")
    ax.set_xlabel("HD cell (sorted)")
    if j == 0:
        ax.set_ylabel("HD cell (sorted)")
fig.colorbar(im, ax=ax, shrink=0.75, label="Pearson r")

ax = fig.add_subplot(gs[1, 0])
ax.plot(pw, pr, ".", ms=4, alpha=0.5, color="darkorange")
lim = max(abs(pw).max(), abs(pr).max()) * 1.1
ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.8)
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_xlabel("wake pairwise corr")
ax.set_ylabel("REM pairwise corr")
ax.set_title(f"wake vs REM: r={r_rem:.2f}, p={p_rem:.4f}")

ax = fig.add_subplot(gs[1, 1])
ax.plot(pw, pn, ".", ms=4, alpha=0.5, color="seagreen")
ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.8)
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_xlabel("wake pairwise corr")
ax.set_ylabel("NREM pairwise corr")
ax.set_title(f"wake vs NREM: r={r_nrem:.2f}, p={p_nrem:.4f}")

ax = fig.add_subplot(gs[1, 2])
ax.hist(null_rem, bins=40, alpha=0.6, color="darkorange", label="REM null")
ax.hist(null_nrem, bins=40, alpha=0.6, color="seagreen", label="NREM null")
ax.axvline(r_rem, color="darkorange", lw=2)
ax.axvline(r_nrem, color="seagreen", lw=2)
ax.set_xlabel("corr-of-corr under label permutation")
ax.set_ylabel("count")
ax.legend(fontsize=8)
ax.set_title("Permutation null")

ax = fig.add_subplot(gs[2, 0])
ax.plot(np.degrees(centers), bw, "o-", color="k", label="wake (active)")
ax.plot(np.degrees(centers), br, "o-", color="darkorange", label="REM")
ax.plot(np.degrees(centers), bn, "o-", color="seagreen", label="NREM")
ax.axhline(0, color="0.7", lw=0.8)
ax.set_xlabel("angular distance between pref. dirs (deg)")
ax.set_ylabel("mean pairwise corr")
ax.legend(fontsize=8)
ax.set_title("Ring metric structure in all states")

ax = fig.add_subplot(gs[2, 1:])
P = post_rem.values.T
ax.imshow(P / (P.sum(axis=0, keepdims=True) + 1e-12), aspect="auto", cmap="magma",
          origin="lower", extent=[dec_rem.t[0], dec_rem.t[-1], 0, 360])
ax.plot(dec_rem.t[enough_rem], np.degrees(dec_rem.values[enough_rem]), ".",
        color="cyan", ms=1.5, alpha=0.35, label="decoded (real, 100 ms bins)")
dv = dec_rem.values.copy()
dv[~enough_rem] = np.nan
kern = np.ones(10) / 10
sm_s = np.convolve(np.nan_to_num(np.sin(dv)), kern, mode="same")
sm_c = np.convolve(np.nan_to_num(np.cos(dv)), kern, mode="same")
cnt = np.convolve((~np.isnan(dv)).astype(float), kern, mode="same")
sm_ang = np.where(cnt > 0.5, np.arctan2(sm_s, sm_c) % (2 * np.pi), np.nan)
sm_deg = np.degrees(sm_ang)
sm_deg[np.abs(np.diff(sm_deg, prepend=sm_deg[0])) > 180] = np.nan
ax.plot(dec_rem.t, sm_deg, color="cyan", lw=1.8, label="decoded (1 s smoothed)")
ax.plot(dec_shuf_rem.t, np.degrees(dec_shuf_rem.values), ".",
        color="white", ms=1, alpha=0.25, label="decoded (time-shifted)")
ax.set_ylabel("virtual HD (deg)")
ax.set_xlabel("time (s)")
ax.set_title("REM episode, best 90 s window: internally maintained bump")
ax.legend(loc="upper right", fontsize=8)
fig.suptitle(f"{name}: ring structure is maintained during sleep", fontsize=13)
fig.savefig("figures/04_sleep_ring.png", dpi=150, bbox_inches="tight")

# ---- figure: bump continuity statistics ----
fig2, axes = plt.subplots(1, 4, figsize=(17, 4))
axes[0].hist(shift_p90, bins=20, color="0.4", alpha=0.8, label="time-shift null")
axes[0].axvline(p90_real, color="darkorange", lw=2, label=f"real {p90_real:.0f} deg/s")
axes[0].set_xlabel("90th percentile |d(decoded HD)/dt| (deg/s)")
axes[0].set_ylabel("count")
axes[0].legend(fontsize=8)
axes[0].set_title(f"Bump drifts slowly (p={p_speed:.3f})")

lags = np.arange(51) * DEC_BIN
axes[1].plot(lags, ac_real, color="darkorange", lw=2, label="real REM")
m = np.nanmean(shift_ac, axis=0)
sd = np.nanstd(shift_ac, axis=0)
axes[1].plot(lags, m, color="0.4", lw=1.5, label="time-shift null")
axes[1].fill_between(lags, m - 2 * sd, m + 2 * sd, color="0.4", alpha=0.3)
axes[1].set_xlabel("lag (s)")
axes[1].set_ylabel("posterior autocorrelation")
axes[1].legend(fontsize=8)
axes[1].set_title(f"Bump persists over seconds (lag-1s p={p_ac10:.3f})")

axes[2].hist(R_wake, bins=50, alpha=0.7, density=True, color="k", label="wake")
axes[2].hist(R_rem, bins=50, alpha=0.7, density=True, color="darkorange", label="REM")
axes[2].set_xlabel("posterior concentration (resultant length)")
axes[2].set_ylabel("density")
axes[2].legend(fontsize=8)
axes[2].set_title("Bump remains localized in REM")

occ_sleep, edges = np.histogram(dec_rem_all.values[enough & ~np.isnan(dec_rem_all.values)],
                                bins=36, range=(0, 2 * np.pi))
axes[3].bar(np.degrees(0.5 * (edges[:-1] + edges[1:])), occ_sleep / occ_sleep.sum(),
            width=8, color="darkorange")
axes[3].set_xlabel("decoded virtual HD (deg)")
axes[3].set_ylabel("occupancy")
axes[3].set_title("REM bump visits the whole ring")
fig2.tight_layout()
fig2.savefig("figures/05_sleep_continuity.png", dpi=150, bbox_inches="tight")

## 6. Replication across sessions

The key analyses (HD cell yield, wake decoding error, correlation
preservation with permutation null, REM bump autocorrelation vs a 20-shift
null) are repeated on four more sessions, one per mouse, and the pairwise
structure is pooled across sessions.

The correlation-structure preservation replicates across sessions (9 of 10
session-by-state tests significant; the exception is the lowest-yield
session). The REM bump-continuity effect (excess posterior autocorrelation
over the conservative rate-preserving time-shift null) is strong in the
session with the largest HD ensemble (Mouse28, 20 HD cells) and weak or
absent in sessions with 5-14 HD cells, where the decoded posterior is
dominated by single-cell slow rate fluctuations that the null also
captures. Detecting the internal bump dynamics against this null requires a
sufficiently large HD ensemble; the correlation preservation above does
not.

In [ ]:
N_SHIFT_PANEL = 20


def analyze_session(name):
    print(f"\n===== {name} =====")
    nwb = load_session(SESSIONS[name])
    units = sorted_units(nwb)
    hd = get_hd_angle(nwb)
    states = get_states(nwb)
    wake = states["Awake"]

    hd_wake = hd.restrict(wake)
    hd_valid_v = hd_wake.values[~np.isnan(hd_wake.values)]
    counts_da = nap.compute_tuning_curves(units, hd, bins=60, range=[(0, 2 * np.pi)],
                                          epochs=wake, return_counts=True,
                                          feature_names=["hd"])
    counts = counts_da.values
    occupancy = counts_da.attrs["occupancy"] / counts_da.attrs["fs"]
    rates = np.where(occupancy[None, :] > 0, counts / occupancy[None, :], np.nan)
    bin_centers = counts_da.coords["hd"].values

    n_units = len(units)
    mvl_obs = np.zeros(n_units)
    p_val = np.ones(n_units)
    pref = np.zeros(n_units)
    for i, k in enumerate(units.keys()):
        spk = units[k].restrict(wake)
        if len(spk) == 0:
            continue
        ang = spk.value_from(hd)
        ang = ang[~np.isnan(ang)]
        if len(ang) > SPIKE_CAP:
            ang = ang[rng.choice(len(ang), SPIKE_CAP, replace=False)]
        if len(ang) < 50:
            continue
        mvl_obs[i] = mean_vector_length(ang)
        pref[i] = np.arctan2(np.mean(np.sin(ang)), np.mean(np.cos(ang))) % (2 * np.pi)
        n = len(ang)
        null = np.zeros(N_SHUFFLE_HD)
        for cs in range(0, N_SHUFFLE_HD, 100):
            mm = min(100, N_SHUFFLE_HD - cs)
            samp = hd_valid_v[rng.integers(0, len(hd_valid_v), size=(mm, n))]
            null[cs:cs + mm] = np.sqrt(np.sin(samp).mean(1) ** 2 + np.cos(samp).mean(1) ** 2)
        p_val[i] = (np.sum(null >= mvl_obs[i]) + 1) / (N_SHUFFLE_HD + 1)
    is_hd = (p_val < 0.05) & (mvl_obs > MVL_FLOOR)
    n_hd = int(is_hd.sum())
    print(f"HD cells: {n_hd}/{n_units}")
    if n_hd < 5:
        return None

    hd_idx = np.where(is_hd)[0]
    hd_sorted = hd_idx[np.argsort(pref[hd_idx])]
    pref_sorted = pref[hd_sorted]
    hd_units = nap.TsGroup({i: units[int(u)] for i, u in enumerate(hd_sorted)})
    iu = np.triu_indices(n_hd, 1)

    tc_rates, _ = flat_tuning(rates[hd_sorted])
    tuning_da = xr.DataArray(tc_rates, dims=("unit", "hd"),
                             coords={"unit": np.arange(n_hd), "hd": bin_centers})
    dec_wake, _ = nap.decode_bayes(tuning_da, hd_units, epochs=wake, bin_size=DEC_BIN)
    actual = dec_wake.value_from(hd)
    valid = ~np.isnan(actual.values) & ~np.isnan(dec_wake.values)
    err = np.angle(np.exp(1j * (dec_wake.values[valid] - actual.values[valid])))
    med_err = np.degrees(np.median(np.abs(err)))

    hd_w = hd.restrict(wake)
    v = np.abs(np.angle(np.exp(1j * np.diff(hd_w.values)))) / np.diff(hd_w.t)
    v = np.concatenate([[0], v])
    v[np.isnan(v)] = 0
    v_sm = gaussian_filter1d(v, sigma=39)
    active_ep = nap.Tsd(t=hd_w.t, d=v_sm).threshold(
        np.percentile(v_sm, 75), "above").time_support.intersect(wake)

    def state_corr(epochs):
        zs, ws = [], []
        for st, en in zip(epochs.start, epochs.end):
            if en - st < 5.0:
                continue
            c = hd_units.count(CORR_BIN, nap.IntervalSet(st, en)).values.astype(float)
            if c.shape[0] < 20:
                continue
            with np.errstate(invalid="ignore"):
                C = np.corrcoef(c.T)
            zs.append(np.arctanh(np.clip(C[iu], -0.999, 0.999)))
            ws.append(c.shape[0])
        Z = np.stack(zs)
        W = np.broadcast_to(np.array(ws, float)[:, None], Z.shape).copy()
        W[np.isnan(Z)] = 0
        Z = np.nan_to_num(Z)
        den = W.sum(0)
        return np.where(den > 0, Z.sum(0) / np.maximum(den, 1e-12), np.nan)

    zw = np.tanh(state_corr(active_ep))
    zr = np.tanh(state_corr(states["REM"]))
    zn = np.tanh(state_corr(states["Non-REM"]))
    fin = np.isfinite(zw) & np.isfinite(zr) & np.isfinite(zn)
    r_rem = np.corrcoef(zw[fin], zr[fin])[0, 1]
    r_nrem = np.corrcoef(zw[fin], zn[fin])[0, 1]
    C_rem = np.full((n_hd, n_hd), np.nan)
    C_rem[iu] = zr
    C_rem[(iu[1], iu[0])] = zr
    C_nrem = np.full((n_hd, n_hd), np.nan)
    C_nrem[iu] = zn
    C_nrem[(iu[1], iu[0])] = zn
    null_rem = np.zeros(N_PERM)
    null_nrem = np.zeros(N_PERM)
    for i in range(N_PERM):
        p = rng.permutation(n_hd)
        null_rem[i] = np.corrcoef(zw[fin], C_rem[np.ix_(p, p)][iu][fin])[0, 1]
        null_nrem[i] = np.corrcoef(zw[fin], C_nrem[np.ix_(p, p)][iu][fin])[0, 1]
    p_rem = (np.sum(null_rem >= r_rem) + 1) / (N_PERM + 1)
    p_nrem = (np.sum(null_nrem >= r_nrem) + 1) / (N_PERM + 1)
    print(f"wake decoding err {med_err:.0f} deg; corr-of-corr REM {r_rem:.2f} "
          f"(p={p_rem:.4f}), NREM {r_nrem:.2f} (p={p_nrem:.4f})")

    rem = states["REM"]
    dec_rem, post_rem = nap.decode_bayes(tuning_da, hd_units, epochs=rem, bin_size=DEC_BIN)
    keep = hd_units.count(DEC_BIN, rem).values.sum(axis=1) >= 2
    ac_real = posterior_autocorr(post_rem, keep)
    ac_null = np.full((N_SHIFT_PANEL, 51), np.nan)
    for i in range(N_SHIFT_PANEL):
        sh = time_shift_group(hd_units, rem, rng)
        _, ps = nap.decode_bayes(tuning_da, sh, epochs=rem, bin_size=DEC_BIN)
        ks = sh.count(DEC_BIN, rem).values.sum(axis=1) >= 2
        ac_null[i] = posterior_autocorr(ps, ks)
    p_ac = (np.nansum(ac_null[:, 10] >= ac_real[10]) + 1) / (N_SHIFT_PANEL + 1)
    print(f"REM autocorr lag-1s: real {ac_real[10]:.3f} vs null "
          f"{np.nanmean(ac_null[:,10]):.3f} (p={p_ac:.3f})")

    return dict(name=name, n_hd=n_hd, n_units=n_units, med_err=med_err,
                r_rem=r_rem, r_nrem=r_nrem, p_rem=p_rem, p_nrem=p_nrem,
                zw=zw[fin], zr=zr[fin], zn=zn[fin],
                d_ang=np.abs(np.angle(np.exp(1j * (pref_sorted[:, None] -
                                                   pref_sorted[None, :]))))[iu][fin],
                ac_real=ac_real, ac_null_mean=np.nanmean(ac_null, axis=0), p_ac=p_ac)


results = []
for pname in ["Mouse25-140123", "Mouse17-130128", "Mouse20-130514", "Mouse24-131213"]:
    res = analyze_session(pname)
    if res is not None:
        results.append(res)

# add the main session
results.append(dict(name=name, n_hd=n_hd, n_units=len(units), med_err=med_err,
                    r_rem=r_rem, r_nrem=r_nrem, p_rem=p_rem, p_nrem=p_nrem,
                    zw=pw, zr=pr, zn=pn, d_ang=d_ang,
                    ac_real=ac_real, ac_null_mean=np.nanmean(shift_ac, axis=0),
                    p_ac=p_ac10))

# cache per-session results so the summary figure can be re-rendered cheaply
for r in results:
    np.savez(f"data/{r['name']}_final_summary.npz",
             **{k: (v if isinstance(v, np.ndarray) else np.asarray(v))
                for k, v in r.items()})

fig, axes = plt.subplots(1, 4, figsize=(17, 4.6))
names = [r["name"] for r in results]
x = np.arange(len(names))
axes[0].bar(x - 0.2, [r["r_rem"] for r in results], width=0.4, color="darkorange",
            label="wake-REM")
axes[0].bar(x + 0.2, [r["r_nrem"] for r in results], width=0.4, color="seagreen",
            label="wake-NREM")


def star(p):
    return "***" if p <= 0.001 else ("**" if p <= 0.01 else ("*" if p <= 0.05 else "n.s."))


for xi, r in zip(x, results):
    axes[0].text(xi - 0.2, r["r_rem"] + 0.04, star(r["p_rem"]), ha="center", fontsize=9)
    axes[0].text(xi + 0.2, r["r_nrem"] + 0.04, star(r["p_nrem"]), ha="center", fontsize=9)
axes[0].set_xticks(x)
axes[0].set_xticklabels([n.replace("-", "\n") for n in names], fontsize=7)
axes[0].set_ylabel("corr-of-corr")
axes[0].set_ylim(0, 1.28)
axes[0].legend(loc="upper center", ncol=2, fontsize=8, framealpha=0.9)
axes[0].set_title("Structure preserved per session\n(*** p<=0.001, ** p<=0.01, * p<=0.05)",
                  fontsize=10)

zw_all = np.concatenate([r["zw"] for r in results])
zr_all = np.concatenate([r["zr"] for r in results])
zn_all = np.concatenate([r["zn"] for r in results])
axes[1].plot(zw_all, zr_all, ".", ms=3, alpha=0.4, color="darkorange", label="REM")
axes[1].plot(zw_all, zn_all, ".", ms=3, alpha=0.4, color="seagreen", label="NREM")
lim = np.percentile(np.abs(np.concatenate([zw_all, zr_all, zn_all])), 99) * 1.2
axes[1].plot([-lim, lim], [-lim, lim], "k--", lw=0.8)
axes[1].set_xlim(-lim, lim)
axes[1].set_ylim(-lim, lim)
r_pool_rem = np.corrcoef(zw_all, zr_all)[0, 1]
r_pool_nrem = np.corrcoef(zw_all, zn_all)[0, 1]
axes[1].set_xlabel("wake pairwise corr")
axes[1].set_ylabel("sleep pairwise corr")
axes[1].legend(fontsize=8, loc="upper left")
axes[1].set_title(f"pooled: REM r={r_pool_rem:.2f}, NREM r={r_pool_nrem:.2f}")

d_all = np.concatenate([r["d_ang"] for r in results])
bid = np.digitize(d_all, bins) - 1
for vals, c, lbl in [(zw_all, "k", "wake (active)"), (zr_all, "darkorange", "REM"),
                     (zn_all, "seagreen", "NREM")]:
    axes[2].plot(np.degrees(centers), [np.mean(vals[bid == b]) for b in range(6)],
                 "o-", color=c, label=lbl)
axes[2].axhline(0, color="0.7", lw=0.8)
axes[2].set_xlabel("angular distance between pref. dirs (deg)")
axes[2].set_ylabel("mean pairwise corr")
axes[2].legend(fontsize=8)
axes[2].set_title("Ring metric structure (pooled)")

for r in results:
    axes[3].plot(lags, r["ac_real"] - r["ac_null_mean"], lw=1.5,
                 label=f"{r['name']} (p={r['p_ac']:.3f})")
axes[3].axhline(0, color="0.5", ls="--", lw=0.8)
axes[3].set_xlabel("lag (s)")
axes[3].set_ylabel("excess posterior autocorrelation\n(real - time-shift null)")
axes[3].legend(fontsize=6.5, loc="upper right")
axes[3].set_title("REM bump persistence above null")
fig.suptitle("Ring attractor maintained during sleep: 5 sessions, 5 mice", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig("figures/06_multisession_summary.png", dpi=150, bbox_inches="tight")
print(f"\npooled corr-of-corr: REM {r_pool_rem:.3f}, NREM {r_pool_nrem:.3f} "
      f"({len(zw_all)} pairs, {len(results)} sessions)")

## Conclusions

- HD cells in the mouse ADn tile the circle of headings with unimodal tuning
  curves; sorted by preferred direction, the population forms a single bump
  of activity that tracks the LED-measured heading. This is the signature of
  a ring attractor read out by the animal's orientation. Wake Bayesian
  decoding accuracy scales with the size of the recorded HD ensemble
  (median error ~20 deg with 20 HD cells in Mouse28; 50-105 deg with 6-14
  HD cells in the other sessions).
- The pairwise correlation pattern of HD cells during wake is preserved
  during both REM and Non-REM sleep across the five sessions tested (one
  per mouse; per-session corr-of-corr 0.49-0.97). Nine of ten
  session-by-state tests are significant against the label-permutation
  null (p <= 0.027); the exception is the NREM test in the session with
  the fewest HD cells (5 cells, i.e. only 10 pairs, where the permutation
  test has the least resolution; p = 0.09). Pooled over 291 pairs,
  corr-of-corr r = 0.90 for REM and 0.84 for NREM. Because sleep removes
  the sensory heading signal, the preserved structure must be maintained
  by internal circuit dynamics. This replicates the central result of
  Peyrache et al. 2015.
- Decoding a virtual heading during REM sleep from the wake tuning curves
  reveals a localized bump (posterior concentration comparable to wake) that
  drifts slowly and visits the whole ring. In the session with the largest
  HD ensemble the bump is significantly more temporally coherent than a
  time-shift control that preserves firing rates and slow single-cell rate
  fluctuations but destroys cross-cell coordination (p90 angular speed
  14 vs 23 deg/s, p = 0.01; posterior autocorrelation at 1 s lag 0.47 vs
  0.41, p = 0.01). In the lower-yield sessions the effect does not reach
  significance against this conservative null, so the continuity result
  rests on the best-recorded session, while the correlation preservation
  is robust across all five.
- Together: the HD system is organized as a one-dimensional ring whose
  structure does not depend on ongoing sensory input, and whose sleep
  activity is consistent with an internally generated bump diffusing along
  the ring, exactly as a continuous ring attractor predicts.